# Python Memory Management
Memory management in Python involves a combination of automatic garbage collection, reference counting, and various internal optimizations to efficiently manage memory allocation and deallocation. Understanding these mechanisms can help developers write more efficient and robust applications.

- Key Concepts in Python Memory Management
- Memory Allocation and Deallocation
- Reference Counting
- Garbage Collection
- The gc Module
- Memory Management Best Practices

### Reference Counting
Reference counting is the primary method Python uses to manage memory. Each object in Python maintains a count of references pointing to it. When the reference count drops to zero, the memory occupied by the object is deallocated.

In [1]:
import sys

a=[]
print(sys.getrefcount(a)) # prints 2, (one from variable a, another from getrefcount() method )

2


In [5]:
b=a
print(sys.getrefcount(a)) # prints 3, (one from variable a, another from variable b, another from getrefcount() method )

3


In [6]:
print(sys.getrefcount(b)) # prints 3, (one from variable a, another from variable b, another from getrefcount() method )

3


In [7]:
del b
print(sys.getrefcount(a)) # prints 2, (one from variable a, another from getrefcount() method )

2


#### Garbage Collection
Python includes a cyclic garbage collector to handle reference cycles. Reference cycles occur when objects reference each other, preventing their reference counts from reaching zero.

In [11]:
import gc # garbage collector
# enable garbage collection
gc.enable()
# disable garbage collection
gc.disable()

# manually run garbage collection
gc.collect() # returns the number of unreachable objects it found and collected

# get garbage collection statistics
print(gc.get_stats()) # returns a list of dictionaries containing statistics about the garbage collection process

# get unreachable objects
print(gc.garbage) # returns a list of objects that are unreachable but cannot be collected

# get garbage collection thresholds
print(gc.get_threshold()) # returns a tuple containing the current collection thresholds for the three generations

[{'collections': 69, 'collected': 2046, 'uncollectable': 0}, {'collections': 6, 'collected': 201, 'uncollectable': 0}, {'collections': 4, 'collected': 751, 'uncollectable': 0}]
[]
(2000, 10, 10)


### Memory Management Best Practices

1. **Use Local Variables**
   Local variables usually have a shorter lifespan than global variables, so they are released from memory sooner.

2. **Avoid Circular References**
   Circular references can make memory cleanup more difficult and may lead to memory leaks if not handled properly.

3. **Use Generators**
   Generators yield one item at a time instead of storing everything in memory at once, which makes them more memory-efficient.

4. **Explicitly Delete Unused Objects**
   Use the `del` statement to remove variables or objects that are no longer needed.

5. **Profile Memory Usage**
   Use tools like `tracemalloc` and `memory_profiler` to detect memory leaks and understand how your program uses memory.


In [ ]:
import gc

class MyObject:
    def __init__(self, name):
        self.name = name
        print(f"Object {self.name} created")

    def __del__(self):
        print(f"Object {self.name} destroyed")

# create circular reference
obj1 = MyObject("A")
obj2 = MyObject("B")
obj1.ref = obj2 # circular reference
obj2.ref = obj1 # circular reference

del obj1 # objects are not garbage collected due to circular reference, deletes the name, not the object itself, the objects remain alive because they still reference each other
del obj2 # objects are not garbage collected due to circular reference, deletes the name, not the object itself, the objects remain alive because they still reference each other

print("Garbage collection stats before collecting:")
print(gc.get_stats())

gc.collect() # manually run garbage collection
print("Garbage collection stats after collecting:")
print(gc.get_stats())

Object A created
Object B created
Garbage collection stats before collecting:
[{'collections': 69, 'collected': 2046, 'uncollectable': 0}, {'collections': 6, 'collected': 201, 'uncollectable': 0}, {'collections': 4, 'collected': 751, 'uncollectable': 0}]
Object A destroyed
Object B destroyed
Garbage collection stats after collecting:
[{'collections': 69, 'collected': 2046, 'uncollectable': 0}, {'collections': 6, 'collected': 201, 'uncollectable': 0}, {'collections': 5, 'collected': 962, 'uncollectable': 0}]


In [13]:
## Generator for memory efficient iteration
# Generators are a type of iterable that generate values on-the-fly, which can be more memory efficient than lists, especially for large datasets.
# Generators allow you to produce items one at a time, using memory efficiently by only keeping one item in memory at a time.

def generate_numbers(n):
    for i in range(n):
        yield i # yield is used to produce a value and pause the function, resuming from the same point when the next value is requested

# using the  generator
for number in generate_numbers(10):
    print(number)
    

0
1
2
3
4
5
6
7
8
9


In [20]:
## Profiling memory usage with tracemalloc
import tracemalloc

def create_large_list(n):
    return [i for i in range(n)]

def main():
    tracemalloc.start() # start tracing memory allocations

    create_large_list(1000000) # create a large list to generate memory usage

    snapshot = tracemalloc.take_snapshot() # take a snapshot of current memory allocations
    top_stats = snapshot.statistics('lineno') # get statistics of memory usage by line number

    print("[ Top 10 ]")
    for stat in top_stats[::]: # print the top 10 memory usage statistics
        print(stat)

In [21]:
main()

[ Top 10 ]
c:\Python313\Lib\json\decoder.py:361: size=2148 B, count=23, average=93 B
c:\Python313\Lib\codeop.py:118: size=1616 B, count=15, average=108 B
C:\Users\sampa\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:1613: size=1520 B, count=1, average=1520 B
C:\Users\sampa\AppData\Roaming\Python\Python313\site-packages\IPython\core\compilerop.py:174: size=1337 B, count=15, average=89 B
C:\Users\sampa\AppData\Roaming\Python\Python313\site-packages\traitlets\traitlets.py:731: size=1192 B, count=19, average=63 B
C:\Users\sampa\AppData\Roaming\Python\Python313\site-packages\traitlets\traitlets.py:1514: size=1080 B, count=9, average=120 B
C:\Users\sampa\AppData\Roaming\Python\Python313\site-packages\zmq\sugar\socket.py:802: size=1056 B, count=6, average=176 B
c:\Python313\Lib\contextlib.py:109: size=944 B, count=10, average=94 B
C:\Users\sampa\AppData\Roaming\Python\Python313\site-packages\zmq\sugar\attrsettr.py:45: size=893 B, count=19, average=47 B
C:\User